# Phase 3: MoE Experts Merge Pipeline

This notebook guides you through merging Task SLMs (LoRA adapters) from Phase 2 into **3 separate Mixture-of-Experts models** - one per organizational unit.

## Overview

- **Input**: 14 Task SLM adapters from Phase 2 (5 + 4 + 5 across 3 units)
- **Output**: 3 separate MoE models:
  - Fundraising MoE (5 experts)
  - Business Development MoE (4 experts)
  - Field Operations MoE (5 experts)
- **Architecture**: Mixtral-style with hidden gate mode
- **Router**: Positive/negative prompt embeddings for semantic routing within each unit
- **Phase 4**: Each MoE powers one A2A agent in the agentic network

## Prerequisites

- GPU runtime (A100 recommended for full merge)
- Phase 2 exports available

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate peft
!pip install -q mergekit
!pip install -q pydantic pydantic-settings pyyaml structlog
!pip install -q sentence-transformers

In [ ]:
# Clone repository (if running in Colab)
import os

if not os.path.exists('phase-3-moe-experts'):
    !git clone https://github.com/your-org/enterprise-ai-habitat.git
    %cd enterprise-ai-habitat/phase-3-moe-experts
else:
    %cd phase-3-moe-experts

In [ ]:
# Add to path
import sys
sys.path.insert(0, '.')

## 2. Configuration

Set up the pipeline configuration for 3 per-unit MoE models.

In [ ]:
from config.settings import get_settings, Settings
from pathlib import Path

# Load settings
settings = get_settings('config/config.yaml')

# Configuration overview
print("MoE Configuration:")
print(f"  Architecture: {settings.moe.architecture}")
print(f"  Gate mode: {settings.moe.gate_mode}")
print(f"  Dtype: {settings.moe.dtype}")
print(f"  Experts per token: {settings.moe.experts_per_token}")
print(f"\nUnits configured: {settings.get_unit_ids()}")
for unit_id in settings.get_unit_ids():
    unit = settings.get_unit(unit_id)
    print(f"  - {unit.name}: {len(unit.tasks)} tasks")

In [ ]:
# Test mode for quick validation (set to False for full merge)
TEST_MODE = True
settings.test_mode = TEST_MODE

if TEST_MODE:
    print("Running in TEST MODE - using mock data (2 experts per unit)")
else:
    print("Running in PRODUCTION MODE - full merge for all 3 units")

## 3. Import Phase 2 Exports

Import and validate the Task SLM adapters from Phase 2, organized by unit.

In [ ]:
from src.program1_import.importer import Phase2AdapterImporter, MockAdapterGenerator
from src.program1_import.validator import ImportValidator, print_validation_report

if TEST_MODE:
    # Generate mock exports for testing (3 units)
    generator = MockAdapterGenerator(settings)
    import_result = generator.generate_mock_exports()
else:
    # Import from Phase 2 exports
    importer = Phase2AdapterImporter(settings)
    import_result = importer.import_adapters()

print(f"\nImported {import_result.total_adapters} adapters across {len(import_result.units)} units")
print(f"Units: {import_result.units}")
print(f"Base model: {import_result.base_model}")

In [ ]:
# Validate imports
validator = ImportValidator(expected_base_model=import_result.base_model)
validation = validator.validate_import_result(import_result)
print_validation_report(validation)

In [ ]:
# List imported adapters by unit
print("\nImported Adapters by Unit:")
for unit_id in sorted(import_result.units):
    unit_adapters = import_result.get_adapters_by_unit(unit_id)
    print(f"\n{unit_id} ({len(unit_adapters)} experts):")
    for adapter in unit_adapters:
        print(f"  - {adapter.task_id}")
        print(f"    Positive prompts: {len(adapter.positive_prompts)}")

## 4. Generate MoE Configurations

Generate mergekit-moe YAML configurations for each unit (3 configs).

In [ ]:
from src.program2_config_gen.mergekit_config import PerUnitMergekitConfigBuilder
import yaml

# Build configs for all units
config_builder = PerUnitMergekitConfigBuilder(settings)

if TEST_MODE:
    configs = config_builder.build_test_configs()
else:
    configs = config_builder.build_all_configs(import_result=import_result)

print(f"Generated {len(configs)} MoE configurations:")
for unit_id, config_path in sorted(configs.items()):
    print(f"  - {unit_id}: {config_path.name}")

In [ ]:
# Display one of the generated configs
sample_unit = list(configs.keys())[0]
with open(configs[sample_unit]) as f:
    config = yaml.safe_load(f)

print(f"Sample MoE Configuration ({sample_unit}):")
print(yaml.dump(config, default_flow_style=False))

In [ ]:
# Generate routing configurations for each unit
from src.program2_config_gen.routing_config import RoutingConfigBuilder

routing_builder = RoutingConfigBuilder(settings)

print("\nGenerating routing configurations:")
for unit_id in import_result.units:
    unit_adapters = import_result.get_adapters_by_unit(unit_id)
    routing_path = routing_builder.build_routing_config(
        adapters=unit_adapters,
        output_filename=f"{unit_id}_routing.json",
        include_embeddings=False
    )
    print(f"  - {unit_id}: {routing_path.name}")

## 5. Execute MoE Merges

Run mergekit-moe to create 3 merged models (one per unit).

**Note**: Full merge requires:
- GPU with sufficient VRAM (A100 40GB recommended)
- Approximately 30-60 minutes per unit

In [ ]:
from src.program3_merge.merger import MoEMerger, MockMerger, check_mergekit_available
from src.shared.config_generator import load_mergekit_config

# Check mergekit availability
if check_mergekit_available():
    print("mergekit-moe is available")
else:
    print("mergekit-moe not found - will use mock merge")

In [ ]:
# Merge all units
base_path = Path('.')
configs_dir = base_path / settings.paths.configs_dir
merged_dir = base_path / settings.paths.merged_dir

merge_results = {}

if TEST_MODE:
    mock_merger = MockMerger(settings)
else:
    merger = MoEMerger(settings)

# Find all unit configs
config_files = list(configs_dir.glob("*_moe.yaml"))

for config_path in sorted(config_files):
    unit_id = config_path.stem.replace("_moe", "")
    output_dir = merged_dir / f"{unit_id}_moe"
    
    print(f"\n--- Merging {unit_id} ---")
    
    config = load_mergekit_config(config_path)
    num_experts = len(config.get("experts", []))
    print(f"Experts: {num_experts}")
    
    if TEST_MODE:
        result = mock_merger.create_mock_merge(
            config_path=config_path,
            output_dir=output_dir
        )
    else:
        result = merger.merge(
            config_path=config_path,
            output_dir=output_dir,
            use_cuda=True
        )
    
    merge_results[unit_id] = result
    status = "SUCCESS" if result.success else "FAILED"
    print(f"Status: {status}")
    if result.duration_seconds:
        print(f"Duration: {result.duration_seconds:.1f}s")

In [ ]:
# Summary of merge results
print("\n" + "=" * 60)
print("Merge Summary:")
print("=" * 60)
for unit_id, result in sorted(merge_results.items()):
    status = "OK" if result.success else "FAIL"
    num_experts = result.metadata.get("num_experts", "?")
    print(f"  {unit_id}: {status} ({num_experts} experts)")
    print(f"    Output: {result.output_dir}")

In [ ]:
# List output files for one unit
sample_unit = list(merge_results.keys())[0]
sample_result = merge_results[sample_unit]

if sample_result.success:
    print(f"\nMerged model files ({sample_unit}):")
    for f in sorted(sample_result.output_dir.rglob('*')):
        if f.is_file():
            print(f"  {f.relative_to(sample_result.output_dir)}")

## 6. Export for Phase 4

Export each unit's MoE model with routing metadata for its A2A agent.

In [ ]:
from src.program5_export.phase4_exporter import Phase4Exporter

exporter = Phase4Exporter(settings)

# Export all units
export_results = exporter.export_all(
    adapters=import_result.adapters,
    generate_agent_configs=True,
    generate_routing_embeddings=not TEST_MODE  # Skip embeddings in test mode
)

print("\n" + "=" * 60)
print("Export Summary:")
print("=" * 60)
for unit_id, result in sorted(export_results.items()):
    status = "OK" if result.success else "FAIL"
    print(f"\n  {unit_id}: {status}")
    print(f"    Export directory: {result.export_dir}")
    print(f"    Model: {'Yes' if result.model_exported else 'No'}")
    print(f"    Routing: {'Yes' if result.routing_exported else 'No'}")
    print(f"    Agent config: {'Yes' if result.agent_config_exported else 'No'}")

In [ ]:
# List exported files for one unit
sample_unit = list(export_results.keys())[0]
sample_export = export_results[sample_unit]

if sample_export.success:
    print(f"\nExport structure ({sample_unit}):")
    for f in sorted(sample_export.export_dir.rglob('*')):
        if f.is_file():
            print(f"  {f.relative_to(sample_export.export_dir)}")

In [ ]:
# Display generated agent config for one unit
import yaml

sample_unit = list(export_results.keys())[0]
agent_config_path = export_results[sample_unit].export_dir / "agent_config" / f"{sample_unit}_agent.yaml"

if agent_config_path.exists():
    with open(agent_config_path) as f:
        agent_config = yaml.safe_load(f)
    
    print(f"Agent Configuration ({sample_unit}):")
    print(yaml.dump(agent_config, default_flow_style=False))

## 7. Verify Merged Models (Optional)

Load and verify one of the merged MoE models.

In [ ]:
# Load and inspect merged model config (production mode only)
if not TEST_MODE:
    import json
    
    for unit_id, result in merge_results.items():
        if result.success:
            config_file = result.output_dir / 'config.json'
            if config_file.exists():
                with open(config_file) as f:
                    model_config = json.load(f)
                
                print(f"\n{unit_id} Model Configuration:")
                print(f"  Model type: {model_config.get('model_type')}")
                print(f"  Num experts: {model_config.get('num_local_experts')}")
                print(f"  Experts per token: {model_config.get('num_experts_per_tok')}")
                print(f"  Hidden size: {model_config.get('hidden_size')}")
else:
    print("Skipping model verification in TEST MODE")

In [ ]:
# Test inference on one unit's MoE (production mode only)
if not TEST_MODE:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    
    # Test the Fundraising MoE
    unit_to_test = "fundraising"
    if unit_to_test in merge_results and merge_results[unit_to_test].success:
        model_path = merge_results[unit_to_test].output_dir
        
        print(f"Loading {unit_to_test} MoE for inference test...")
        model = AutoModelForCausalLM.from_pretrained(
            str(model_path),
            torch_dtype='auto',
            device_map='auto'
        )
        tokenizer = AutoTokenizer.from_pretrained(str(model_path))
        
        # Test prompt (matched to this unit's domain)
        prompt = "Profile the investor: John Smith is a venture capitalist specializing in"
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True
        )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"\nPrompt: {prompt}")
        print(f"Response: {response}")
else:
    print("Skipping inference test in TEST MODE")

## 8. Summary

Pipeline complete! The 3 merged MoE models are ready for Phase 4 A2A integration.

In [ ]:
print("=" * 60)
print("Phase 3: MoE Merge Pipeline Complete")
print("=" * 60)
print(f"\nMode: {'TEST' if TEST_MODE else 'PRODUCTION'}")
print(f"Units processed: {len(merge_results)}")
print(f"Total adapters: {import_result.total_adapters}")

print("\nMerge Results:")
for unit_id, result in sorted(merge_results.items()):
    status = "OK" if result.success else "FAIL"
    print(f"  {unit_id}: {status}")

print("\nExport Results:")
for unit_id, result in sorted(export_results.items()):
    status = "OK" if result.success else "FAIL"
    print(f"  {unit_id}: {status}")

print("\nExport Locations:")
for unit_id, result in sorted(export_results.items()):
    print(f"  {unit_id}: {result.export_dir}")

print("\nNext steps:")
print("  1. Copy each unit's export to its corresponding Phase 4 A2A agent")
print("  2. Configure agent routing using exported agent_config/*.yaml")
print("  3. Start A2A agent services (one per unit)")
print("\nPhase 4 Architecture:")
print("  Fundraising MoE    → Fundraising A2A Agent")
print("  Business Development MoE → Business Development A2A Agent")
print("  Field Operations MoE     → Field Operations A2A Agent")
print("=" * 60)